In [3]:
import os,random
import numpy as np
import pandas as pd                                           
from sklearn.metrics import roc_auc_score,accuracy_score, cohen_kappa_score     # Metrics

from keras.models import load_model # Model loading Method

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import img_to_array, load_img

from skimage import io
from skimage import feature, color
from skimage.filters import roberts, sobel, scharr, prewitt

2025-04-09 11:39:20.421590: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-09 11:39:20.468530: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-09 11:39:20.474615: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-09 11:39:30.017131: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
## ----------------------------------------------------------------- Custom preprocessing wrapper function -------------------------------------------------------- #
def preprocessing_wrapper(method='canny'):
    def preprocessing_function(image):
        return perform_edge_detection(image, method=method)
    return preprocessing_function

def perform_edge_detection(image,method):
    image = color.rgb2gray(image)    # Convert to grayscale
    
    if method == 'canny':
        image = feature.canny(image, sigma=0.02).astype(np.float32)
    if method == 'sobel':
        image = sobel(image).astype(np.float32)
    if method == 'roberts':
        image = roberts(image).astype(np.float32)
    if method == 'scharr':
        image = scharr(image).astype(np.float32)
    if method == 'prewitt':
        image = prewitt(image).astype(np.float32)
        
    image = np.expand_dims(image, axis=-1)  # Add channel dimension
    image = np.repeat(image, 3, axis=-1)    # Repeat the single channel to create a 3-channel image
    return image

In [55]:
###------------------------------------------FAIRY-CIRCLES-------------------------------------#

# seed used in training
seed = 42

# Set the image size and batch size
image_size = (224, 224)  # Inception V3 input size
batch_size = 32

dataset_dir = '/app/External Images/other_shape_external_set'

# Prepare data using ImageDataGenerator with validation split
datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create the validation generator
external_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',
    seed = seed,
    shuffle=False
)


Found 60 images belonging to 1 classes.


In [6]:
alexnet_model_path   = '/app/model-weights/RGB-Based-CNN/alexnet_results/alexnet_best_weights.tf' # Load your trained CNN model
resnet_model_path    = '/app/model-weights/RGB-Based-CNN/resnet_results/resnet_best_weights.tf' # Load your trained CNN model
inception_model_path = '/app/model-weights/RGB-Based-CNN/inceptionv3_results/inception_best_weights.tf'
vgg_model_path       = '/app/model-weights/RGB-Based-CNN/vgg16_results/vgg_best_weights.tf'

In [7]:
rgb_model_to_weights_path = dict()
    
rgb_model_to_weights_path['VGG16'] = vgg_model_path
rgb_model_to_weights_path['InceptionV3'] = inception_model_path
rgb_model_to_weights_path['Alexnet'] = alexnet_model_path
rgb_model_to_weights_path['ReNnet50'] = resnet_model_path

rgb_model_to_weights_path

{'VGG16': '/app/model-weights/RGB-Based-CNN/vgg16_results/vgg_best_weights.tf',
 'InceptionV3': '/app/model-weights/RGB-Based-CNN/inceptionv3_results/inception_best_weights.tf',
 'Alexnet': '/app/model-weights/RGB-Based-CNN/alexnet_results/alexnet_best_weights.tf',
 'ReNnet50': '/app/model-weights/RGB-Based-CNN/resnet_results/resnet_best_weights.tf'}

In [33]:
alexnet_model = load_model(alexnet_model_path)
resnet_model = load_model(resnet_model_path)
inception_model = load_model(inception_model_path)
vgg_model = load_model(vgg_model_path)

rgb_algo_to_model_path = {
    'alexnet_model':alexnet_model,
    'resnet_model':resnet_model,
    'inception_model':inception_model,
    'vgg_model':vgg_model
}

In [41]:
# Make predictions
alexnet_y_val_prob = alexnet_model.predict(external_generator)
resne_y_val_prob = resnet_model.predict(external_generator)
inception_y_val_prob = inception_model.predict(external_generator)
vgg_y_val_prob = vgg_model.predict(external_generator)

2/2 [==============================] - 2s 808ms/step


In [42]:
print(f'AlexNet\t-->\tFairy Circle Predicted : {len(alexnet_y_val_prob[alexnet_y_val_prob < 0.5])}')
print(f'ResNet50\t-->\tFairy Circle Predicted : {len(resne_y_val_prob[resne_y_val_prob < 0.5])}')
print(f'InceptionV3\t--->\tFairy Circle Predicted : {len(resne_y_val_prob[inception_y_val_prob < 0.5])}')
print(f'VGG16\t--->\tFairy Circle Predicted : {len(vgg_y_val_prob[vgg_y_val_prob < 0.5])}')

AlexNet	-->	Fairy Circle Predicted : 6
ResNet50	-->	Fairy Circle Predicted : 38
InceptionV3	--->	Fairy Circle Predicted : 40
VGG16	--->	Fairy Circle Predicted : 14


In [28]:
def get_prediction_and_corresponding_file_path(algo,alexnet_y_val_prob,external_generator):
    # 2. If it's binary classification (class_mode='binary'), convert probabilities to labels
    predicted_labels = (alexnet_y_val_prob > 0.5).astype(int).flatten()
    
    # 3. Get full file paths
    filepaths = external_generator.filepaths
    full_filepaths = [ os.path.join(external_generator.directory, path) for path in filepaths]
    
    # 4. Combine file path with predicted label
    predicted_data = list(zip(full_filepaths, alexnet_y_val_prob[:,0], predicted_labels))
    predicted_data = pd.DataFrame(predicted_data,columns = ['path', 'pred_prob','label'])
    predicted_data['Algorithm'] = algo
    predicted_data = predicted_data[['Algorithm' , 'path', 'pred_prob','label']]
    return predicted_data

In [37]:
all_algo_prediction_df = pd.DataFrame()
for algo, algo_model in rgb_algo_to_model_path.items():
    pred_prob = algo_model.predict(external_generator)
    predicted_data = get_prediction_and_corresponding_file_path(algo, pred_prob, external_generator)
    all_algo_prediction_df = pd.concat([all_algo_prediction_df,predicted_data], axis=0)


2/2 [==============================] - 3s 764ms/step


In [44]:
# all_algo_prediction_df[
#     (all_algo_prediction_df['label'] == 0) & 
#     (all_algo_prediction_df['Algorithm'] == 'alexnet_model')
# ]


In [50]:
# pip install openpyxl

In [51]:
all_algo_prediction_df.to_excel('RGB_MODELS_EXTERNAL_EVALUATIONS.xlsx',index=False)

In [52]:
pwd

'/app'

# EDGE BASED

In [32]:
edge_based_weights_dir = '/external/model-weights/EDGE-Based-CNN'

all_files = os.listdir(edge_based_weights_dir)
substrings = ['canny', 
              'sobel','roberts']
weights_dirs = [
    file for file in all_files if any(sub in file for sub in substrings) and file.endswith('.py') !=True and file.endswith('results') ==True
]


# cwd = os.getcwd()
model_to_weights_path = dict()
for model in weights_dirs:
    model_path = os.path.join(edge_based_weights_dir,model)
    model_to_weights_path[model] = model_path

edge_model_to_weights_path = model_to_weights_path
model_to_weights_path

{'alexnet_canny_results': '/external/model-weights/EDGE-Based-CNN/alexnet_canny_results',
 'alexnet_roberts_results': '/external/model-weights/EDGE-Based-CNN/alexnet_roberts_results',
 'alexnet_sobel_results': '/external/model-weights/EDGE-Based-CNN/alexnet_sobel_results',
 'inceptionv3_canny_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_canny_results',
 'inceptionv3_roberts_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_roberts_results',
 'inceptionv3_sobel_results': '/external/model-weights/EDGE-Based-CNN/inceptionv3_sobel_results',
 'resnet50_canny_results': '/external/model-weights/EDGE-Based-CNN/resnet50_canny_results',
 'resnet50_roberts_results': '/external/model-weights/EDGE-Based-CNN/resnet50_roberts_results',
 'resnet50_sobel_results': '/external/model-weights/EDGE-Based-CNN/resnet50_sobel_results',
 'vgg16_canny_results': '/external/model-weights/EDGE-Based-CNN/vgg16_canny_results',
 'vgg16_roberts_results': '/external/model-weights/EDGE-Based-

In [54]:
all_edge_algo_prediction_df = pd.DataFrame()
model_to_weights_path = edge_model_to_weights_path

for cnn_algo, model_path in model_to_weights_path.items():
    
    model_dir_contents = os.listdir(model_to_weights_path[cnn_algo])
    
    weights_file = [file for file in model_dir_contents if file.endswith('.tf')][0]
    
    weights_file_path = os.path.join(model_to_weights_path[cnn_algo],weights_file)
    pred_prob = model.predict(external_generator)
    
    predicted_data = get_prediction_and_corresponding_file_path( model_name, pred_prob, external_generator)
    all_edge_algo_prediction_df = pd.concat([ all_edge_algo_prediction_df, predicted_data], axis=0)

AttributeError: 'str' object has no attribute 'predict'